# Local Hydrological Stress Model — H_local(x, t)

## Modele

Vecteur de contrainte hydrologique local :

$$H_{local}(x,t) = \big(\, b(x),\; g(x),\; d(x,t),\; S(x,t),\; V(x,t),\; U(x,t) \,\big)$$

| Variable | Symbole | Nature | Source |
|---|---|---|---|
| Baseline Water Stress | $b(x)$ | structurel, surfacique | Aqueduct 4.0 BWS |
| Groundwater Stress | $g(x)$ | structurel, souterrain | Aqueduct 4.0 GWS |
| Drought Indicator | $d(x,t)$ | dynamique | ERA5 → SPI-12 |
| Persistence | $S(x,t)$ | dynamique | moyenne glissante 3 mois de $d$ |
| Variability | $V(x,t)$ | dynamique | ecart-type glissant 12 mois de $d$ |
| Uncertainty | $U(x,t)$ | meta | proxy fiabilite donnees |

Indice local :

$$L(x,t) = w_b \cdot b(x) + w_g \cdot g(x) + w_d \cdot d(x,t) + w_S \cdot S(x,t) + w_V \cdot V(x,t) + w_U \cdot U(x,t)$$

Toutes les variables sont normalisees dans $[0, 1]$ : **0 = favorable, 1 = contrainte forte**.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

# ── Chemins donnees ──────────────────────────────────────────────────
BASE = Path("../water_model/data")
RAW  = BASE / "raw"

AQUEDUCT_BWS = RAW / "aqueduct" / "bws_raw.tif"
AQUEDUCT_GWS = RAW / "aqueduct" / "gws_raw.tif"
AQUEDUCT_DRR = RAW / "aqueduct" / "drr_raw.tif"
ERA5_FULL    = RAW / "era5" / "era5_monthly_1981_2022.nc"

# ── Point test : Paris-Saclay (zone datacenter) ─────────────────────
TEST_LAT = 48.73
TEST_LON = 2.17
TEST_DATE = "2022-07"

print(f"Point test : ({TEST_LAT}, {TEST_LON})")
print(f"Date test  : {TEST_DATE}")
print(f"BWS raster : {AQUEDUCT_BWS.exists()}")
print(f"GWS raster : {AQUEDUCT_GWS.exists()}")
print(f"ERA5 serie : {ERA5_FULL.exists()}")

---
## 1. Extraction BWS / GWS au point (lat, lon)

Source : **WRI Aqueduct 4.0** — Hofste et al. (2019), update Kuzma et al. (2023)

- BWS = Baseline Water Stress = withdrawals / available supply (0–5 scale)
- GWS = Groundwater Table Decline → converti en 0–5 par Aqueduct

On extrait la valeur brute au pixel le plus proche du point (lat, lon).
La resolution Aqueduct est ~0.01° (~1 km), donc le nearest-neighbor est suffisant.

In [ ]:
def extract_raster_point(raster_path: Path, lat: float, lon: float) -> float:
    """
    Extrait la valeur d'un raster GeoTIFF au point (lat, lon).
    
    Utilise rasterio.sample() qui fait du nearest-neighbor sur le pixel
    contenant le point. Si le point tombe sur un pixel nodata, retourne NaN.
    
    Args:
        raster_path: chemin vers le .tif (EPSG:4326 attendu)
        lat, lon: coordonnees WGS84
    
    Returns:
        valeur du pixel (float), ou NaN si nodata
    """
    with rasterio.open(raster_path) as src:
        # rasterio.sample attend (lon, lat) = (x, y)
        vals = list(src.sample([(lon, lat)]))
        val = float(vals[0][0])
        
        # Masquer nodata
        if src.nodata is not None and val == src.nodata:
            return np.nan
        return val


# ── Extraction ───────────────────────────────────────────────────────
bws_raw = extract_raster_point(AQUEDUCT_BWS, TEST_LAT, TEST_LON)
gws_raw = extract_raster_point(AQUEDUCT_GWS, TEST_LAT, TEST_LON)
drr_raw = extract_raster_point(AQUEDUCT_DRR, TEST_LAT, TEST_LON)

print(f"BWS raw = {bws_raw:.4f}  (echelle 0-5, max observe en France: ~3.0)")
print(f"GWS raw = {gws_raw:.4f}  (echelle 0-5, max observe en France: ~0.8)")
print(f"DRR raw = {drr_raw:.4f}  (echelle 0-1, Aqueduct drought risk natif)")
print()

# Verification : le point est-il dans l'emprise ?
with rasterio.open(AQUEDUCT_BWS) as src:
    b = src.bounds
    in_bounds = (b.left <= TEST_LON <= b.right) and (b.bottom <= TEST_LAT <= b.top)
    print(f"Emprise raster : [{b.left}, {b.bottom}] -> [{b.right}, {b.top}]")
    print(f"Point dans emprise : {in_bounds}")
    # Resolution effective
    res_x, res_y = src.res
    print(f"Resolution : {res_x:.4f}° x {res_y:.4f}° ({res_x*111:.1f} km x {res_y*111:.1f} km)")

---
## 2. Chargement ERA5 — serie de precipitations au point

Source : **ECMWF ERA5 Reanalysis** via Copernicus CDS

- Variable : `tp` (total precipitation) en **m/jour** (monthly mean rate)
- Conversion en mm/mois : `tp × jours_du_mois × 1000`
- Resolution : 0.25° (~28 km) — on prend le gridpoint le plus proche
- Serie : 1981-01 → 2022-12 (504 mois, 42 ans)

In [ ]:
def load_era5_precip_at_point(
    nc_path: Path,
    lat: float,
    lon: float,
) -> tuple[np.ndarray, pd.DatetimeIndex, float, float, float]:
    """
    Charge la serie ERA5 et extrait les precipitations mensuelles au point.
    
    Returns:
        precip_mm : array (N,) precipitations en mm/mois
        dates     : DatetimeIndex correspondant
        era5_lat  : latitude reelle du gridpoint ERA5
        era5_lon  : longitude reelle du gridpoint ERA5
        dist_km   : distance point demande <-> gridpoint ERA5 (km)
    """
    ds = xr.open_dataset(nc_path)
    
    # ERA5 utilise "valid_time" comme coordonnee temporelle
    time_dim = "valid_time" if "valid_time" in ds.dims else "time"
    
    # Nearest gridpoint
    tp_point = ds["tp"].sel(latitude=lat, longitude=lon, method="nearest")
    era5_lat = float(tp_point.latitude)
    era5_lon = float(tp_point.longitude)
    
    # Distance au gridpoint (Haversine simplifie)
    dlat = np.radians(lat - era5_lat)
    dlon = np.radians(lon - era5_lon)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat)) * np.cos(np.radians(era5_lat)) * np.sin(dlon/2)**2
    dist_km = 6371 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    
    # Conversion m/jour -> mm/mois
    times = pd.DatetimeIndex(tp_point[time_dim].values)
    days_in_month = times.days_in_month.values.astype(float)
    tp_values = tp_point.values.astype(float)
    precip_mm = tp_values * days_in_month * 1000.0
    
    # Clamp negatifs (artefacts numeriques ERA5)
    precip_mm = np.maximum(precip_mm, 0.0)
    
    ds.close()
    return precip_mm, times, era5_lat, era5_lon, dist_km


# ── Extraction ───────────────────────────────────────────────────────
precip_mm, dates, era5_lat, era5_lon, dist_km = load_era5_precip_at_point(
    ERA5_FULL, TEST_LAT, TEST_LON
)

print(f"Gridpoint ERA5 : ({era5_lat:.2f}, {era5_lon:.2f})")
print(f"Distance au point demande : {dist_km:.1f} km")
print(f"Serie : {dates[0].strftime('%Y-%m')} -> {dates[-1].strftime('%Y-%m')} ({len(dates)} mois)")
print(f"Precip mm/mois : min={precip_mm.min():.1f}, max={precip_mm.max():.1f}, mean={precip_mm.mean():.1f}")
print(f"Precip annuelle moyenne : {precip_mm.mean() * 12:.0f} mm/an")

---
## 3. Calcul SPI-12 au point

**Standardized Precipitation Index** — McKee, Doesken & Kleist (1993)

Methode :
1. **Accumulation glissante** : $P_{12}(t) = \sum_{i=t-11}^{t} P(i)$ (somme 12 mois)
2. **Fit gamma** sur la periode de reference 1981–2010 :
   - Distribution $\Gamma(\alpha, \beta)$ avec $\text{loc}=0$ (precipitation $\geq 0$)
   - Traitement des zeros : masse de probabilite discrete $q_0$ (Thom, 1966)
3. **Transformation** : $p = q_0 + (1-q_0) \cdot F_\Gamma(P_{12})$ puis $\text{SPI} = \Phi^{-1}(p)$

Le SPI est **adimensionnel**, centre sur 0 par construction.

| SPI | Interpretation (OMM) |
|-----|---------------------|
| $\geq 2.0$ | Extremement humide |
| $[-1, 1]$ | Normal |
| $\leq -1.0$ | Moderement sec |
| $\leq -2.0$ | Extremement sec |

In [ ]:
def compute_spi_1d(
    precip_mm: np.ndarray,
    dates: pd.DatetimeIndex,
    window: int = 12,
    baseline_start: int = 1981,
    baseline_end: int = 2010,
) -> dict:
    """
    Calcule le SPI-N pour une serie temporelle 1D de precipitations.
    
    Implementation fidele a McKee et al. (1993) avec correction de
    Thom (1966) pour les zeros.
    
    Args:
        precip_mm    : precipitations mensuelles en mm (N,)
        dates        : index temporel correspondant
        window       : fenetre d'accumulation en mois (12 = SPI-12)
        baseline_start, baseline_end : periode de reference pour le fit gamma
    
    Returns:
        dict avec :
            spi          : array (N,) valeurs SPI (NaN pour les window-1 premiers mois)
            precip_acc   : array (N,) precipitations accumulees sur window mois
            gamma_alpha  : parametre shape du fit gamma
            gamma_beta   : parametre scale du fit gamma
            gamma_pvalue : p-value du test KS (qualite du fit)
            q0           : proportion de zeros dans la baseline
            n_baseline   : nombre de mois baseline valides
    """
    n = len(precip_mm)
    
    # ── Etape 1 : accumulation glissante ─────────────────────────────
    precip_acc = np.full(n, np.nan)
    for i in range(window - 1, n):
        precip_acc[i] = np.sum(precip_mm[i - window + 1 : i + 1])
    
    # ── Etape 2 : separer baseline et cible ──────────────────────────
    years = dates.year
    baseline_mask = (years >= baseline_start) & (years <= baseline_end)
    baseline_acc = precip_acc[baseline_mask]
    baseline_valid = baseline_acc[~np.isnan(baseline_acc)]
    n_baseline = len(baseline_valid)
    
    if n_baseline < 30:
        raise ValueError(
            f"Baseline insuffisante : {n_baseline} mois valides "
            f"(minimum 30 requis pour un fit gamma fiable)"
        )
    
    # ── Etape 3 : proportion de zeros (Thom, 1966) ──────────────────
    q0 = np.sum(baseline_valid == 0.0) / n_baseline
    baseline_pos = baseline_valid[baseline_valid > 0.0]
    
    if len(baseline_pos) < 10:
        raise ValueError(
            f"Pas assez de mois avec precipitation positive : {len(baseline_pos)}"
        )
    
    # ── Etape 4 : fit gamma (floc=0, contrainte physique P >= 0) ─────
    alpha, _loc, beta = stats.gamma.fit(baseline_pos, floc=0.0)
    
    # ── Etape 5 : test de Kolmogorov-Smirnov (qualite du fit) ────────
    ks_stat, ks_pvalue = stats.kstest(baseline_pos, "gamma", args=(alpha, 0.0, beta))
    
    # ── Etape 6 : transformation SPI ─────────────────────────────────
    spi = np.full(n, np.nan)
    
    for i in range(n):
        p_acc = precip_acc[i]
        if np.isnan(p_acc):
            continue
        
        if p_acc == 0.0:
            # Masse de probabilite discrete pour les zeros
            prob = q0
        else:
            # Probabilite composite : P(X <= p) = q0 + (1-q0) * Gamma_cdf(p)
            prob = q0 + (1.0 - q0) * stats.gamma.cdf(p_acc, alpha, loc=0.0, scale=beta)
        
        # Clamper pour eviter les infinis dans ppf
        prob = np.clip(prob, 1e-6, 1.0 - 1e-6)
        
        # SPI = quantile normal standard
        spi[i] = stats.norm.ppf(prob)
    
    return {
        "spi": spi,
        "precip_acc": precip_acc,
        "gamma_alpha": alpha,
        "gamma_beta": beta,
        "gamma_pvalue": ks_pvalue,
        "q0": q0,
        "n_baseline": n_baseline,
    }


# ── Calcul ───────────────────────────────────────────────────────────
spi_result = compute_spi_1d(precip_mm, dates, window=12)

spi_series = spi_result["spi"]
spi_valid = spi_series[~np.isnan(spi_series)]

print("=== Fit Gamma (baseline 1981-2010) ===")
print(f"  alpha (shape)  = {spi_result['gamma_alpha']:.4f}")
print(f"  beta  (scale)  = {spi_result['gamma_beta']:.2f}")
print(f"  q0 (frac zeros)= {spi_result['q0']:.4f}")
print(f"  n baseline     = {spi_result['n_baseline']}")
print(f"  KS p-value     = {spi_result['gamma_pvalue']:.4f}  {'OK' if spi_result['gamma_pvalue'] > 0.05 else 'ATTENTION: fit mediocre'}")
print()
print("=== SPI-12 statistiques ===")
print(f"  N valides  = {len(spi_valid)}")
print(f"  Moyenne    = {spi_valid.mean():.3f}  (attendu ~0)")
print(f"  Ecart-type = {spi_valid.std():.3f}  (attendu ~1)")
print(f"  Min / Max  = [{spi_valid.min():.2f}, {spi_valid.max():.2f}]")
print(f"  % sec (<-1)     = {100 * np.sum(spi_valid < -1) / len(spi_valid):.1f}%")
print(f"  % tres sec(<-2) = {100 * np.sum(spi_valid < -2) / len(spi_valid):.1f}%")

---
## 4. Feature Engineering

### 4.1 b(x) — Baseline Water Stress normalise

Transformation : normalisation lineaire sur l'echelle Aqueduct 0–5.

$$b(x) = \text{clip}\!\left(\frac{\text{BWS}_{raw}(x)}{5.0},\; 0,\; 1\right)$$

**Hypothese** : la relation BWS → stress est lineaire sur [0, 5]. C'est une simplification
acceptable en v1 — pour une version plus fine, utiliser la normalisation sigmoide de
Pfister et al. (2009) qui sature aux extremes.

### 4.2 g(x) — Groundwater Stress normalise

$$g(x) = \text{clip}\!\left(\frac{\text{GWS}_{raw}(x)}{5.0},\; 0,\; 1\right)$$

Meme logique que b(x). Note : le GWS Aqueduct en France est generalement faible (< 0.8),
ce qui est coherent — les nappes francaises ne sont pas en situation de depletion severe
comparees a l'Inde ou la Californie.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 4.1  b(x) — Baseline Water Stress normalise
# ══════════════════════════════════════════════════════════════════════

def compute_bws_norm(bws_raw: float, bws_max: float = 5.0) -> float:
    """
    Normalise le BWS brut Aqueduct en [0, 1].
    
    Transformation lineaire : b(x) = clip(BWS_raw / BWS_max, 0, 1)
    
    Echelle Aqueduct 4.0 (Kuzma et al. 2023) :
        0   = pas de stress (prelevements negligeables vs disponibilite)
        1   = stress faible  (20% des ressources prelevees)
        2-3 = stress modere a eleve
        5   = stress extreme (prelevements >= offre)
    
    Args:
        bws_raw : valeur BWS brute [0, 5]
        bws_max : borne superieure de l'echelle (defaut 5.0)
    
    Returns:
        b(x) dans [0, 1]
    """
    if np.isnan(bws_raw):
        return np.nan
    return float(np.clip(bws_raw / bws_max, 0.0, 1.0))


# ══════════════════════════════════════════════════════════════════════
# 4.2  g(x) — Groundwater Stress normalise
# ══════════════════════════════════════════════════════════════════════

def compute_gws_norm(gws_raw: float, gws_max: float = 5.0) -> float:
    """
    Normalise le GWS brut Aqueduct en [0, 1].
    
    Transformation lineaire : g(x) = clip(GWS_raw / GWS_max, 0, 1)
    
    Le GWS Aqueduct represente le declin de la nappe phreatique.
    En France metropolitaine, les valeurs sont generalement < 1.0
    (pas de depletion severe a l'echelle nationale).
    
    Args:
        gws_raw : valeur GWS brute [0, 5]
        gws_max : borne superieure de l'echelle (defaut 5.0)
    
    Returns:
        g(x) dans [0, 1]
    """
    if np.isnan(gws_raw):
        return np.nan
    return float(np.clip(gws_raw / gws_max, 0.0, 1.0))


# ── Calcul ───────────────────────────────────────────────────────────
b_x = compute_bws_norm(bws_raw)
g_x = compute_gws_norm(gws_raw)

print(f"b(x) = {b_x:.4f}  (BWS normalise, raw={bws_raw:.3f}/5)")
print(f"g(x) = {g_x:.4f}  (GWS normalise, raw={gws_raw:.3f}/5)")

### 4.3 d(x,t) — Drought Indicator

Transformation du SPI en indicateur de secheresse normalise $[0, 1]$ :

$$d(x,t) = \text{clip}\!\left(\frac{-\text{SPI}(x,t)}{3},\; 0,\; 1\right)$$

| SPI | d(x,t) | Interpretation |
|-----|--------|----------------|
| $\geq 0$ | 0 | Pas de stress (normal ou humide) |
| $-1$ | 0.33 | Secheresse moderee |
| $-2$ | 0.67 | Secheresse severe |
| $\leq -3$ | 1.0 | Secheresse extreme |

**Choix du seuil** : SPI = -3 comme borne maximale car les valeurs < -3 sont
statistiquement rares ($<0.1\%$) et ne changent pas qualitativement le diagnostic.

### 4.4 S(x,t) — Persistence de la secheresse

Moyenne glissante de $d$ sur 3 mois :

$$S(x,t) = \frac{1}{3}\sum_{k=0}^{2} d(x, t-k)$$

**Interpretation physique** : une secheresse de 1 mois est un evenement ponctuel ;
3 mois consecutifs de $d > 0.3$ indiquent une secheresse installee qui impacte
les reserves (sols, nappes superficielles). $S$ capture cette inertie.

### 4.5 V(x,t) — Variabilite temporelle

Ecart-type glissant de $d$ sur 12 mois, normalise :

$$V(x,t) = \min\!\left(\frac{\sigma_{12\text{mois}}(d)}{0.5},\; 1\right)$$

Le diviseur $0.5$ correspond a l'ecart-type maximal theorique d'une variable $[0,1]$
(atteint quand la moitie des valeurs sont 0 et l'autre moitie 1).

**Interpretation physique** : $V$ eleve = regime erratique, alternance rapide
sec/humide. $V$ faible = regime stable (soit constamment sec, soit constamment humide).
Un $V$ eleve est defavorable car il rend la planification plus incertaine.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 4.3  d(x,t) — Drought Indicator
# ══════════════════════════════════════════════════════════════════════

def compute_drought_series(spi: np.ndarray, spi_extreme: float = 3.0) -> np.ndarray:
    """
    Convertit une serie SPI en indicateur de secheresse [0, 1].
    
    Formule : d = clip(-SPI / spi_extreme, 0, 1)
    
    Ref: adaptation de la classification OMM (2012).
    Le seuil spi_extreme=3 correspond au cas extreme (~0.1% de probabilite).
    
    Args:
        spi          : serie SPI (N,)
        spi_extreme  : SPI borne basse (defaut 3.0 → SPI=-3 donne d=1)
    
    Returns:
        d : serie drought indicator (N,) dans [0, 1]
    """
    return np.clip(-spi / spi_extreme, 0.0, 1.0)


# ══════════════════════════════════════════════════════════════════════
# 4.4  S(x,t) — Persistence de la secheresse
# ══════════════════════════════════════════════════════════════════════

def compute_persistence(drought: np.ndarray, window: int = 3) -> np.ndarray:
    """
    Moyenne glissante de d sur `window` mois.
    
    S(x,t) = mean(d(t), d(t-1), ..., d(t-window+1))
    
    Interpretation physique :
        S ~ 0   : pas de secheresse persistante
        S > 0.3 : secheresse installee depuis >= 3 mois
        S > 0.6 : secheresse persistante severe
    
    Note : S est naturellement dans [0,1] car d est dans [0,1].
    Les NaN sont propages — un seul NaN dans la fenetre donne NaN.
    
    Args:
        drought : serie d(x,t) (N,)
        window  : taille de la fenetre (defaut 3 mois)
    
    Returns:
        S : serie persistence (N,)
    """
    n = len(drought)
    S = np.full(n, np.nan)
    for i in range(window - 1, n):
        chunk = drought[i - window + 1 : i + 1]
        if np.any(np.isnan(chunk)):
            continue
        S[i] = np.mean(chunk)
    return S


# ══════════════════════════════════════════════════════════════════════
# 4.5  V(x,t) — Variabilite temporelle
# ══════════════════════════════════════════════════════════════════════

def compute_variability(drought: np.ndarray, window: int = 12) -> np.ndarray:
    """
    Ecart-type glissant de d sur `window` mois, normalise par 0.5.
    
    V(x,t) = min(std(d sur window mois) / 0.5, 1)
    
    Le diviseur 0.5 est l'ecart-type maximal theorique d'une variable [0,1]
    (atteint quand P(X=0) = P(X=1) = 0.5).
    
    Interpretation physique :
        V ~ 0   : regime stable (constamment sec OU constamment humide)
        V ~ 0.5 : variabilite moderee (alternance saisonniere normale)
        V > 0.7 : regime tres erratique (risque planification)
    
    Args:
        drought : serie d(x,t) (N,)
        window  : fenetre (defaut 12 mois)
    
    Returns:
        V : serie variabilite (N,) dans [0, 1]
    """
    n = len(drought)
    V = np.full(n, np.nan)
    max_std = 0.5  # ecart-type max theorique d'une variable [0, 1]
    for i in range(window - 1, n):
        chunk = drought[i - window + 1 : i + 1]
        valid = chunk[~np.isnan(chunk)]
        if len(valid) < window // 2:  # au moins 6 mois valides sur 12
            continue
        V[i] = min(np.std(valid, ddof=1) / max_std, 1.0)
    return V


# ── Calcul sur la serie complete ─────────────────────────────────────
drought_series = compute_drought_series(spi_series)
persistence_series = compute_persistence(drought_series, window=3)
variability_series = compute_variability(drought_series, window=12)

# ── Extraire au mois cible ───────────────────────────────────────────
target_idx = None
target_date = pd.Timestamp(TEST_DATE)
for i, d in enumerate(dates):
    if d.year == target_date.year and d.month == target_date.month:
        target_idx = i
        break

if target_idx is None:
    print(f"ERREUR: date {TEST_DATE} non trouvee dans la serie")
else:
    d_xt = drought_series[target_idx]
    S_xt = persistence_series[target_idx]
    V_xt = variability_series[target_idx]
    
    print(f"=== Features dynamiques au {TEST_DATE} ===")
    print(f"  SPI-12         = {spi_series[target_idx]:+.3f}")
    print(f"  d(x,t)         = {d_xt:.4f}  (drought indicator)")
    print(f"  S(x,t)         = {S_xt:.4f}  (persistence 3 mois)")
    print(f"  V(x,t)         = {V_xt:.4f}  (variabilite 12 mois)")
    print()
    
    # Contexte : 3 derniers mois de d
    print("  Contexte (3 derniers mois de d) :")
    for k in range(3):
        idx = target_idx - k
        if idx >= 0 and not np.isnan(drought_series[idx]):
            print(f"    {dates[idx].strftime('%Y-%m')} : d = {drought_series[idx]:.4f}")


### 4.6 U(x,t) — Proxy d'incertitude

$U$ n'est pas un signal physique mais un **meta-indicateur** qui quantifie la fiabilite
des 5 autres variables. Il est compose de 4 sous-composantes :

$$U(x,t) = 0.30 \cdot u_{temp} + 0.25 \cdot u_{struct} + 0.25 \cdot u_{fit} + 0.20 \cdot u_{spatial}$$

| Composante | Signification | Calcul |
|---|---|---|
| $u_{temp}$ | Completude temporelle | fraction de NaN dans les 12 derniers mois du SPI |
| $u_{struct}$ | Disponibilite donnees structurelles | 0.5 si BWS manquant + 0.5 si GWS manquant |
| $u_{fit}$ | Qualite du fit gamma | $\text{clip}(1 - 10 \cdot p_{KS},\; 0,\; 1)$ |
| $u_{spatial}$ | Proximite ERA5 | $\text{clip}(d_{km} / 14,\; 0,\; 1)$, ou 14 km = demi-resolution ERA5 |

**Convention** : $U = 0$ = donnees fiables, $U = 1$ = donnees non fiables.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 4.6  U(x,t) — Proxy d'incertitude
# ══════════════════════════════════════════════════════════════════════

def compute_uncertainty(
    spi: np.ndarray,
    target_idx: int,
    bws_raw: float,
    gws_raw: float,
    gamma_pvalue: float,
    era5_dist_km: float,
    spi_window: int = 12,
) -> tuple[float, dict]:
    """
    Calcule le proxy d'incertitude U(x,t) et ses composantes.
    
    4 sous-composantes, ponderees :
        u_temporal   (0.30) : fraction de NaN dans les 12 derniers mois du SPI
        u_structural (0.25) : disponibilite des donnees BWS/GWS
        u_fit        (0.25) : qualite du fit gamma (via p-value KS)
        u_spatial    (0.20) : distance au gridpoint ERA5
    
    Args:
        spi          : serie SPI complete (N,)
        target_idx   : index du mois cible dans la serie
        bws_raw      : valeur BWS brute (NaN si manquant)
        gws_raw      : valeur GWS brute (NaN si manquant)
        gamma_pvalue : p-value du test KS sur le fit gamma
        era5_dist_km : distance au gridpoint ERA5 en km
        spi_window   : fenetre SPI (defaut 12)
    
    Returns:
        (U, components) ou U est dans [0, 1] et components est un dict detaille
    """
    # ── u_temporal : completude des 12 derniers mois ─────────────────
    start = max(0, target_idx - spi_window + 1)
    window_spi = spi[start : target_idx + 1]
    n_nan = np.sum(np.isnan(window_spi))
    n_total = len(window_spi)
    u_temporal = n_nan / max(n_total, 1)
    
    # ── u_structural : donnees Aqueduct disponibles ? ────────────────
    u_structural = 0.0
    if np.isnan(bws_raw):
        u_structural += 0.5
    if np.isnan(gws_raw):
        u_structural += 0.5
    
    # ── u_fit : qualite du fit gamma ─────────────────────────────────
    # p-value > 0.10 : bon fit → u_fit ~ 0
    # p-value < 0.01 : mauvais fit → u_fit ~ 0.9
    u_fit = float(np.clip(1.0 - 10.0 * gamma_pvalue, 0.0, 1.0))
    
    # ── u_spatial : distance au gridpoint ERA5 ───────────────────────
    # ERA5 resolution = 0.25° ~ 28 km. Demi-resolution = 14 km.
    # Si distance = 0 → u_spatial = 0. Si distance >= 14 km → u_spatial = 1.
    u_spatial = float(np.clip(era5_dist_km / 14.0, 0.0, 1.0))
    
    # ── Agregation ponderee ──────────────────────────────────────────
    U = (0.30 * u_temporal
       + 0.25 * u_structural
       + 0.25 * u_fit
       + 0.20 * u_spatial)
    
    components = {
        "u_temporal": round(u_temporal, 4),
        "u_structural": round(u_structural, 4),
        "u_fit": round(u_fit, 4),
        "u_spatial": round(u_spatial, 4),
    }
    
    return float(np.clip(U, 0.0, 1.0)), components


# ── Calcul ───────────────────────────────────────────────────────────
U_xt, U_components = compute_uncertainty(
    spi=spi_series,
    target_idx=target_idx,
    bws_raw=bws_raw,
    gws_raw=gws_raw,
    gamma_pvalue=spi_result["gamma_pvalue"],
    era5_dist_km=dist_km,
)

print(f"=== U(x,t) — Incertitude au {TEST_DATE} ===")
print(f"  U(x,t) = {U_xt:.4f}")
print()
print(f"  Composantes :")
print(f"    u_temporal   = {U_components['u_temporal']:.4f}  (completude SPI window)")
print(f"    u_structural = {U_components['u_structural']:.4f}  (dispo BWS/GWS)")
print(f"    u_fit        = {U_components['u_fit']:.4f}  (qualite fit gamma, KS p={spi_result['gamma_pvalue']:.4f})")
print(f"    u_spatial    = {U_components['u_spatial']:.4f}  (dist ERA5 = {dist_km:.1f} km)")

---
## 5. Assemblage H_local(x,t) et calcul L(x,t)

### Vecteur de contrainte

$$H_{local}(x,t) = (b, g, d, S, V, U) \in [0,1]^6$$

### Indice local

$$L(x,t) = \sum_i w_i \cdot H_i(x,t), \quad \sum_i w_i = 1$$

**Poids par defaut** :

| Variable | Poids | Justification |
|---|---|---|
| $b$ (BWS) | 0.25 | Indicateur structurel principal (prelevements vs offre) |
| $g$ (GWS) | 0.15 | Composante souterraine, moins critique en France |
| $d$ (drought) | 0.25 | Signal temporel principal (secheresse en cours) |
| $S$ (persistence) | 0.15 | Amplifie le signal si secheresse installee |
| $V$ (variability) | 0.10 | Penalise l'imprevisibilite |
| $U$ (uncertainty) | 0.10 | Discount factor — prudence quand donnees fragiles |

Les poids sont **configurables**. La seule contrainte est $\sum w_i = 1$.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 5.1  Vecteur de contrainte H_local(x,t)
# ══════════════════════════════════════════════════════════════════════

from dataclasses import dataclass

@dataclass
class HLocal:
    """Vecteur de contrainte hydrologique local — 6 dimensions."""
    b: float    # BWS normalise [0,1]
    g: float    # GWS normalise [0,1]
    d: float    # drought indicator [0,1]
    S: float    # persistence [0,1]
    V: float    # variabilite [0,1]
    U: float    # incertitude [0,1]
    
    def as_array(self) -> np.ndarray:
        return np.array([self.b, self.g, self.d, self.S, self.V, self.U])
    
    def as_dict(self) -> dict:
        return {
            "b_bws": self.b, "g_gws": self.g, "d_drought": self.d,
            "S_persistence": self.S, "V_variability": self.V, "U_uncertainty": self.U,
        }
    
    def __repr__(self):
        return (f"H_local(b={self.b:.3f}, g={self.g:.3f}, d={self.d:.3f}, "
                f"S={self.S:.3f}, V={self.V:.3f}, U={self.U:.3f})")


# ══════════════════════════════════════════════════════════════════════
# 5.2  Poids configurables
# ══════════════════════════════════════════════════════════════════════

DEFAULT_WEIGHTS = {
    "w_b": 0.25,   # BWS
    "w_g": 0.15,   # GWS
    "w_d": 0.25,   # drought
    "w_S": 0.15,   # persistence
    "w_V": 0.10,   # variability
    "w_U": 0.10,   # uncertainty
}


# ══════════════════════════════════════════════════════════════════════
# 5.3  Calcul de L(x,t)
# ══════════════════════════════════════════════════════════════════════

def compute_local_index(H: HLocal, weights: dict = None) -> tuple[float, dict]:
    """
    Calcule l'indice de contrainte local L(x,t).
    
    L = w_b*b + w_g*g + w_d*d + w_S*S + w_V*V + w_U*U
    
    Args:
        H       : vecteur de contrainte HLocal
        weights : dict de poids (defaut DEFAULT_WEIGHTS). Doit sommer a 1.
    
    Returns:
        (L, detail) ou L est dans [0, 1] et detail contient les contributions
    
    Raises:
        ValueError si les poids ne somment pas a 1 (tolerance 1e-3)
    """
    w = weights or DEFAULT_WEIGHTS
    
    # Validation
    w_sum = sum(w.values())
    if abs(w_sum - 1.0) > 1e-3:
        raise ValueError(f"Les poids doivent sommer a 1.0, got {w_sum:.4f}")
    
    h = H.as_array()
    w_arr = np.array([w["w_b"], w["w_g"], w["w_d"], w["w_S"], w["w_V"], w["w_U"]])
    
    # Gestion des NaN : si une composante est NaN, on redistribue son poids
    # proportionnellement aux composantes valides
    nan_mask = np.isnan(h)
    if np.all(nan_mask):
        return np.nan, {"error": "toutes les composantes sont NaN"}
    
    if np.any(nan_mask):
        # Redistribution
        w_valid = w_arr.copy()
        w_valid[nan_mask] = 0.0
        w_valid = w_valid / w_valid.sum()  # re-normaliser
        h_safe = np.where(nan_mask, 0.0, h)
        L = float(np.dot(w_valid, h_safe))
        redistribution = True
    else:
        L = float(np.dot(w_arr, h))
        w_valid = w_arr
        redistribution = False
    
    # Contributions individuelles
    contributions = {
        "b_contrib": float(w_valid[0] * (h[0] if not nan_mask[0] else 0)),
        "g_contrib": float(w_valid[1] * (h[1] if not nan_mask[1] else 0)),
        "d_contrib": float(w_valid[2] * (h[2] if not nan_mask[2] else 0)),
        "S_contrib": float(w_valid[3] * (h[3] if not nan_mask[3] else 0)),
        "V_contrib": float(w_valid[4] * (h[4] if not nan_mask[4] else 0)),
        "U_contrib": float(w_valid[5] * (h[5] if not nan_mask[5] else 0)),
        "redistributed": redistribution,
    }
    
    return float(np.clip(L, 0.0, 1.0)), contributions


# ══════════════════════════════════════════════════════════════════════
# 5.4  Assemblage final
# ══════════════════════════════════════════════════════════════════════

H = HLocal(b=b_x, g=g_x, d=d_xt, S=S_xt, V=V_xt, U=U_xt)
L, detail = compute_local_index(H)

print(f"{'='*60}")
print(f" H_local({TEST_LAT}, {TEST_LON}, {TEST_DATE})")
print(f"{'='*60}")
print(f"  b(x)   = {H.b:.4f}   [BWS]")
print(f"  g(x)   = {H.g:.4f}   [GWS]")
print(f"  d(x,t) = {H.d:.4f}   [drought]")
print(f"  S(x,t) = {H.S:.4f}   [persistence]")
print(f"  V(x,t) = {H.V:.4f}   [variability]")
print(f"  U(x,t) = {H.U:.4f}   [uncertainty]")
print(f"{'─'*60}")
print(f"  L(x,t) = {L:.4f}")
print(f"{'─'*60}")
print(f"  Contributions :")
names = ["b(BWS)", "g(GWS)", "d(drought)", "S(persist)", "V(variab)", "U(uncert)"]
keys  = ["b_contrib", "g_contrib", "d_contrib", "S_contrib", "V_contrib", "U_contrib"]
for name, key in zip(names, keys):
    bar = "█" * int(detail[key] / L * 30) if L > 0 else ""
    print(f"    {name:14s} : {detail[key]:.4f}  ({detail[key]/L*100:.1f}%)  {bar}")
print(f"{'='*60}")

---
## 6. Fonction wrapper : `compute_local_stress(lat, lon, date)`

Pipeline complet en un seul appel. Charge les donnees une fois (cache), puis requete rapide par point.

In [ ]:
class LocalStressEngine:
    """
    Moteur de calcul du stress hydrologique local.
    
    Charge les donnees une seule fois a l'initialisation,
    puis compute(lat, lon, date) est rapide (~50ms par point).
    
    Usage:
        engine = LocalStressEngine(bws_path, gws_path, era5_path)
        result = engine.compute(48.73, 2.17, "2022-07")
        print(result["L"], result["H"])
    """
    
    def __init__(
        self,
        bws_path: Path,
        gws_path: Path,
        era5_path: Path,
        spi_window: int = 12,
        baseline_start: int = 1981,
        baseline_end: int = 2010,
        weights: dict = None,
    ):
        self.bws_path = bws_path
        self.gws_path = gws_path
        self.spi_window = spi_window
        self.baseline_start = baseline_start
        self.baseline_end = baseline_end
        self.weights = weights or DEFAULT_WEIGHTS
        
        # Pre-charger ERA5 (le plus lourd)
        ds = xr.open_dataset(era5_path)
        self._era5_tp = ds["tp"]
        time_dim = "valid_time" if "valid_time" in ds.dims else "time"
        self._time_dim = time_dim
        self._era5_lats = ds.latitude.values
        self._era5_lons = ds.longitude.values
        
        # Cache par gridpoint ERA5
        self._spi_cache: dict[tuple[float, float], dict] = {}
    
    def compute(self, lat: float, lon: float, date: str) -> dict:
        """
        Calcule H_local(x,t) et L(x,t) pour un point et une date.
        
        Args:
            lat, lon : coordonnees WGS84
            date     : format "YYYY-MM" (ex: "2022-07")
        
        Returns:
            dict avec L, H, detail, spi_result, debug
        """
        # ── 1. Extraction Aqueduct ───────────────────────────────────
        bws_raw = extract_raster_point(self.bws_path, lat, lon)
        gws_raw = extract_raster_point(self.gws_path, lat, lon)
        
        b = compute_bws_norm(bws_raw)
        g = compute_gws_norm(gws_raw)
        
        # ── 2. ERA5 + SPI (avec cache par gridpoint) ────────────────
        spi_data = self._get_spi_at_point(lat, lon)
        spi_series = spi_data["spi"]
        dates_idx = spi_data["dates"]
        
        # ── 3. Trouver l'index de la date cible ─────────────────────
        target = pd.Timestamp(date)
        target_idx = None
        for i, dt in enumerate(dates_idx):
            if dt.year == target.year and dt.month == target.month:
                target_idx = i
                break
        
        if target_idx is None:
            raise ValueError(f"Date {date} non trouvee dans la serie ERA5")
        
        # ── 4. Features dynamiques ───────────────────────────────────
        drought = compute_drought_series(spi_series)
        d = float(drought[target_idx]) if not np.isnan(drought[target_idx]) else np.nan
        
        persistence = compute_persistence(drought, window=3)
        S = float(persistence[target_idx]) if not np.isnan(persistence[target_idx]) else np.nan
        
        variability = compute_variability(drought, window=12)
        V = float(variability[target_idx]) if not np.isnan(variability[target_idx]) else np.nan
        
        U, u_comp = compute_uncertainty(
            spi=spi_series, target_idx=target_idx,
            bws_raw=bws_raw, gws_raw=gws_raw,
            gamma_pvalue=spi_data["gamma_pvalue"],
            era5_dist_km=spi_data["dist_km"],
        )
        
        # ── 5. Assemblage ────────────────────────────────────────────
        H = HLocal(b=b, g=g, d=d, S=S, V=V, U=U)
        L, contributions = compute_local_index(H, self.weights)
        
        return {
            "L": L,
            "H": H,
            "contributions": contributions,
            "raw": {"bws": bws_raw, "gws": gws_raw},
            "spi_at_t": float(spi_series[target_idx]) if not np.isnan(spi_series[target_idx]) else np.nan,
            "uncertainty_detail": u_comp,
            "era5_gridpoint": (spi_data["era5_lat"], spi_data["era5_lon"]),
        }
    
    def _get_spi_at_point(self, lat: float, lon: float) -> dict:
        """Calcule le SPI au gridpoint ERA5 le plus proche, avec cache."""
        # Trouver le gridpoint ERA5 le plus proche
        lat_idx = int(np.argmin(np.abs(self._era5_lats - lat)))
        lon_idx = int(np.argmin(np.abs(self._era5_lons - lon)))
        era5_lat = float(self._era5_lats[lat_idx])
        era5_lon = float(self._era5_lons[lon_idx])
        
        cache_key = (era5_lat, era5_lon)
        if cache_key in self._spi_cache:
            return self._spi_cache[cache_key]
        
        # Extraire la serie de precipitations
        tp_point = self._era5_tp.isel(latitude=lat_idx, longitude=lon_idx)
        times = pd.DatetimeIndex(tp_point[self._time_dim].values)
        days_in_month = times.days_in_month.values.astype(float)
        precip_mm = np.maximum(tp_point.values.astype(float) * days_in_month * 1000.0, 0.0)
        
        # Distance
        dlat = np.radians(lat - era5_lat)
        dlon = np.radians(lon - era5_lon)
        a = np.sin(dlat/2)**2 + np.cos(np.radians(lat)) * np.cos(np.radians(era5_lat)) * np.sin(dlon/2)**2
        dist_km = 6371 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
        
        # SPI
        spi_result = compute_spi_1d(
            precip_mm, times,
            window=self.spi_window,
            baseline_start=self.baseline_start,
            baseline_end=self.baseline_end,
        )
        
        result = {
            **spi_result,
            "dates": times,
            "precip_mm": precip_mm,
            "era5_lat": era5_lat,
            "era5_lon": era5_lon,
            "dist_km": dist_km,
        }
        self._spi_cache[cache_key] = result
        return result


# ── Instanciation ────────────────────────────────────────────────────
engine = LocalStressEngine(
    bws_path=AQUEDUCT_BWS,
    gws_path=AQUEDUCT_GWS,
    era5_path=ERA5_FULL,
)

# ── Test rapide ──────────────────────────────────────────────────────
result = engine.compute(TEST_LAT, TEST_LON, TEST_DATE)
print(f"compute_local_stress({TEST_LAT}, {TEST_LON}, '{TEST_DATE}')")
print(f"  → L = {result['L']:.4f}")
print(f"  → H = {result['H']}")

---
## 7. Validation spatiale — comparaison multi-sites

Test de coherence : on attend un **gradient Nord-Ouest → Sud-Est** du stress hydrique,
avec Marseille/Montpellier plus stresses que Brest/Rennes.

Sites selectionnes pres de datacenters existants ou projetes.

In [ ]:
SITES = {
    "Paris-Saclay":  (48.73, 2.17),
    "Lyon":          (45.76, 4.83),
    "Marseille":     (43.30, 5.37),
    "Strasbourg":    (48.57, 7.75),
    "Brest":         (48.39, -4.49),
    "Rennes":        (48.11, -1.68),
    "Bordeaux":      (44.84, -0.58),
    "Montpellier":   (43.61, 3.88),
}

DATE = "2022-07"  # plein ete — stress maximal attendu

rows = []
for name, (lat, lon) in SITES.items():
    r = engine.compute(lat, lon, DATE)
    row = {
        "Site": name,
        "lat": lat, "lon": lon,
        "b(x)": r["H"].b,
        "g(x)": r["H"].g,
        "d(x,t)": r["H"].d,
        "S(x,t)": r["H"].S,
        "V(x,t)": r["H"].V,
        "U(x,t)": r["H"].U,
        "L(x,t)": r["L"],
        "SPI-12": r["spi_at_t"],
    }
    rows.append(row)

df = pd.DataFrame(rows).set_index("Site")

# Affichage
print(f"=== Comparaison multi-sites — {DATE} ===\n")
display_cols = ["b(x)", "g(x)", "d(x,t)", "S(x,t)", "V(x,t)", "U(x,t)", "L(x,t)", "SPI-12"]
print(df[display_cols].round(3).to_string())
print()

# Validation : gradient attendu
L_brest = df.loc["Brest", "L(x,t)"]
L_marseille = df.loc["Marseille", "L(x,t)"]
print(f"Gradient NW→SE : L(Brest)={L_brest:.3f} vs L(Marseille)={L_marseille:.3f}")
if L_marseille > L_brest:
    print("  ✓ Coherent — Marseille plus stresse que Brest")
else:
    print("  ✗ ATTENTION — gradient inverse, a investiguer")

---
## 8. Variation temporelle — L(x,t) sur 12 mois

Evolution du vecteur de contrainte pour Paris-Saclay sur l'annee 2022.
On attend un pic de stress en ete (juillet-aout) quand la secheresse s'installe.

In [ ]:
# Evolution temporelle sur 2022 pour Paris-Saclay
months_2022 = [f"2022-{m:02d}" for m in range(1, 13)]

temporal_rows = []
for m in months_2022:
    r = engine.compute(TEST_LAT, TEST_LON, m)
    temporal_rows.append({
        "Mois": m,
        "b": r["H"].b, "g": r["H"].g,
        "d": r["H"].d, "S": r["H"].S,
        "V": r["H"].V, "U": r["H"].U,
        "L": r["L"],
    })

df_temporal = pd.DataFrame(temporal_rows).set_index("Mois")

print(f"=== Paris-Saclay ({TEST_LAT}, {TEST_LON}) — 2022 mensuel ===\n")
print(df_temporal.round(4).to_string())
print()
print(f"L min = {df_temporal['L'].min():.4f} ({df_temporal['L'].idxmin()})")
print(f"L max = {df_temporal['L'].max():.4f} ({df_temporal['L'].idxmax()})")
print(f"L ete (JJA) = {df_temporal.loc[['2022-06','2022-07','2022-08'], 'L'].mean():.4f}")
print(f"L hiver (DJF) = {df_temporal.loc[['2022-01','2022-02','2022-12'], 'L'].mean():.4f}")

---
## 9. Diagnostic de redondance — le probleme S/V/U

**Hypothese a tester** : S, V, U sont derives du meme signal SPI que d.
Si la correlation est forte, on a une reparametrisation, pas un enrichissement.

Tests :
1. **Matrice de correlation** sur les series temporelles multi-sites
2. **VIF** (Variance Inflation Factor) — VIF > 5 = colinearite problematique
3. **Ablation study** — retirer chaque variable et mesurer le delta sur L

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 9.1  Construction du dataset multi-sites x multi-dates
# ══════════════════════════════════════════════════════════════════════

# On calcule H_local pour tous les sites x tous les mois 2022
all_rows = []
for name, (lat, lon) in SITES.items():
    for m in range(1, 13):
        date_str = f"2022-{m:02d}"
        r = engine.compute(lat, lon, date_str)
        all_rows.append({
            "site": name, "date": date_str,
            "b": r["H"].b, "g": r["H"].g,
            "d": r["H"].d, "S": r["H"].S,
            "V": r["H"].V, "U": r["H"].U,
            "L": r["L"],
        })

df_all = pd.DataFrame(all_rows)
print(f"Dataset : {len(df_all)} observations ({len(SITES)} sites x 12 mois)")
print()

# ══════════════════════════════════════════════════════════════════════
# 9.2  Matrice de correlation
# ══════════════════════════════════════════════════════════════════════

features = ["b", "g", "d", "S", "V", "U"]
corr = df_all[features].corr()

print("=== Matrice de correlation ===")
print(corr.round(3).to_string())
print()

# Couples fortement correles (|r| > 0.7)
print("Couples avec |r| > 0.70 :")
found = False
for i, f1 in enumerate(features):
    for j, f2 in enumerate(features):
        if j > i and abs(corr.loc[f1, f2]) > 0.70:
            print(f"  {f1} <-> {f2} : r = {corr.loc[f1, f2]:.3f}  ← REDONDANCE")
            found = True
if not found:
    print("  Aucun (toutes les correlations < 0.70)")
print()

# ══════════════════════════════════════════════════════════════════════
# 9.3  VIF (Variance Inflation Factor)
# ══════════════════════════════════════════════════════════════════════

def compute_vif(df_features: pd.DataFrame) -> pd.Series:
    """
    VIF = 1 / (1 - R^2) pour chaque variable regressee sur les autres.
    VIF > 5 = colinearite problematique.
    VIF > 10 = colinearite severe.
    """
    from numpy.linalg import lstsq
    cols = df_features.columns.tolist()
    # Drop rows with NaN
    df_clean = df_features.dropna()
    vifs = {}
    for col in cols:
        y = df_clean[col].values
        X = df_clean[[c for c in cols if c != col]].values
        X = np.column_stack([np.ones(len(X)), X])
        coef, _, _, _ = lstsq(X, y, rcond=None)
        y_pred = X @ coef
        ss_res = np.sum((y - y_pred)**2)
        ss_tot = np.sum((y - y.mean())**2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        vifs[col] = 1 / (1 - r2) if r2 < 1 else np.inf
    return pd.Series(vifs)

vif = compute_vif(df_all[features])
print("=== Variance Inflation Factor ===")
print("(VIF > 5 = colinearite, VIF > 10 = severe)")
print()
for f in features:
    flag = " ← PROBLEME" if vif[f] > 5 else ""
    print(f"  {f:4s} : VIF = {vif[f]:.2f}{flag}")
print()

# ══════════════════════════════════════════════════════════════════════
# 9.4  Ablation study
# ══════════════════════════════════════════════════════════════════════

print("=== Ablation Study ===")
print("On retire chaque variable et on mesure l'impact sur L.")
print()

# L complet
L_full = df_all["L"].values

# Pour chaque variable, recalculer L sans elle
feature_keys = ["w_b", "w_g", "w_d", "w_S", "w_V", "w_U"]
feature_names = ["b", "g", "d", "S", "V", "U"]

for i, (fkey, fname) in enumerate(zip(feature_keys, feature_names)):
    # Poids sans cette variable (redistribues)
    w_ablated = DEFAULT_WEIGHTS.copy()
    removed_weight = w_ablated.pop(fkey)
    # Re-normaliser
    total = sum(w_ablated.values())
    w_ablated = {k: v / total for k, v in w_ablated.items()}
    # Re-ajouter la variable retiree avec poids 0
    w_ablated[fkey] = 0.0
    # Reordonner
    w_ordered = {k: w_ablated[k] for k in feature_keys}
    
    # Recalculer L pour chaque observation
    L_ablated = []
    for _, row in df_all.iterrows():
        H_row = HLocal(b=row["b"], g=row["g"], d=row["d"],
                       S=row["S"], V=row["V"], U=row["U"])
        L_val, _ = compute_local_index(H_row, w_ordered)
        L_ablated.append(L_val)
    L_ablated = np.array(L_ablated)
    
    # Impact
    delta_mean = np.nanmean(L_full - L_ablated)
    delta_std = np.nanstd(L_full - L_ablated)
    corr_with_full = np.corrcoef(
        L_full[~np.isnan(L_full) & ~np.isnan(L_ablated)],
        L_ablated[~np.isnan(L_full) & ~np.isnan(L_ablated)]
    )[0, 1]
    
    print(f"  Sans {fname:4s} (w={removed_weight:.2f}) : "
          f"delta_L = {delta_mean:+.4f} +/- {delta_std:.4f}, "
          f"corr(L_full, L_ablated) = {corr_with_full:.4f}")

print()
print("Si corr ~ 1.0 : retirer la variable ne change presque rien → redondante.")
print("Si delta_L ~ 0 : la variable ne contribue pas au score → inutile.")

In [ ]:
# Liberer la memoire du moteur v1 avant v2
del engine
import gc
gc.collect()
print('Memoire v1 liberee')


---
## 10. Modele v2 — SPEI (physiquement fonde)

Le SPI ne capture que le cote **offre** (precipitations). Le stress reel depend du bilan :

$$\text{Stress} \sim P - PET$$

Le **SPEI** (Vicente-Serrano et al., 2010) standardise $D = P - PET$ sur 12 mois.
La PET est deja dans notre fichier ERA5 (variable `pev`).

### Nouveau modele

$$H_{local}(x,t) = (b, g, d_{SPEI}, U) \qquad L = 0.30b + 0.20g + 0.40d + 0.10U$$

- S et V supprimes (redondants, cf. section 9)
- d passe du SPI au **SPEI** (information physique supplementaire)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 10.1  PET au point + SPEI-12 (fit par mois calendaire)
# ══════════════════════════════════════════════════════════════════════
#
# Correction majeure vs v2 : on ajuste une distribution log-logistique
# PAR MOIS calendaire (12 fits), et non une seule sur toute la serie.
# Raison : la distribution de D_acc = sum(P-PET, 12 mois) a une forme
# differente en janvier (fin d'hiver, recharge) vs juillet (deficit max).
# Un fit global melange ces regimes et biaise le SPEI.
#
# Ref: Vicente-Serrano et al. (2010), J. Climate, 23(7), 1696-1718.
#      Begueria et al. (2014), Int. J. Climatol., 34, 3001-3023.

def load_era5_pet_at_point(nc_path, lat, lon):
    ds = xr.open_dataset(nc_path)
    td = 'valid_time' if 'valid_time' in ds.dims else 'time'
    pev = ds['pev'].sel(latitude=lat, longitude=lon, method='nearest')
    times = pd.DatetimeIndex(pev[td].values)
    days = times.days_in_month.values.astype(float)
    pet_mm = np.maximum(-pev.values.astype(float) * days * 1000.0, 0.0)
    ds.close()
    return pet_mm, times

def compute_spei_1d(precip_mm, pet_mm, dates, window=12,
                    baseline_start=1981, baseline_end=2010):
    """
    SPEI-N avec fit log-logistique PAR MOIS calendaire.
    
    Pour chaque mois m (1..12) :
      1. Extraire D_acc[m] sur la baseline
      2. Shift pour rendre positif
      3. Fit fisk(c, loc=0, scale) sur baseline[m]
      4. Transformer toute la serie[m] via norm.ppf(fisk.cdf(...))
    
    Retourne un dict avec spei, D_monthly, D_acc, et diagnostics par mois.
    """
    D = precip_mm - pet_mm
    n = len(D)
    
    # Accumulation glissante N mois
    D_acc = np.full(n, np.nan)
    for i in range(window - 1, n):
        D_acc[i] = np.sum(D[i - window + 1 : i + 1])
    
    years = dates.year
    months = dates.month
    mask_base = (years >= baseline_start) & (years <= baseline_end)
    
    # Fit par mois calendaire
    monthly_params = {}  # {month: (c, scale, shift, ks_pval, n_base)}
    
    for m in range(1, 13):
        mask_m = (months == m) & mask_base & ~np.isnan(D_acc)
        base_m = D_acc[mask_m]
        
        if len(base_m) < 10:
            monthly_params[m] = None
            continue
        
        shift_m = -np.min(base_m) + 1.0
        shifted = base_m + shift_m
        
        try:
            c, _, scale = stats.fisk.fit(shifted, floc=0)
            ks_stat, ks_pval = stats.kstest(shifted, 'fisk', args=(c, 0, scale))
            monthly_params[m] = (c, scale, shift_m, ks_pval, len(base_m))
        except Exception:
            monthly_params[m] = None
    
    # Appliquer la transformation par mois
    spei = np.full(n, np.nan)
    for i in range(n):
        if np.isnan(D_acc[i]):
            continue
        m = months[i]
        params = monthly_params.get(m)
        if params is None:
            continue
        c, scale, shift_m, _, _ = params
        prob = stats.fisk.cdf(D_acc[i] + shift_m, c, loc=0, scale=scale)
        prob = np.clip(prob, 1e-6, 1 - 1e-6)
        spei[i] = stats.norm.ppf(prob)
    
    # KS pvalue moyenne (diagnostic global)
    ks_pvals = [p[3] for p in monthly_params.values() if p is not None]
    mean_ks = np.mean(ks_pvals) if ks_pvals else 0.0
    
    return {
        'spei': spei, 'D_monthly': D, 'D_acc': D_acc,
        'ks_pvalue': mean_ks,
        'monthly_params': monthly_params,
        'n_months_fitted': sum(1 for p in monthly_params.values() if p is not None),
    }

# ── Calcul sur Paris-Saclay ────────────────────────────────────────
pet_mm, _ = load_era5_pet_at_point(ERA5_FULL, TEST_LAT, TEST_LON)
spei_result = compute_spei_1d(precip_mm, pet_mm, dates)
spei_series = spei_result['spei']

print(f'SPEI-12 (fit par mois calendaire) — Paris-Saclay')
print(f'Mois fittes : {spei_result["n_months_fitted"]}/12')
print(f'KS pvalue moyen : {spei_result["ks_pvalue"]:.4f}')
print()

# Diagnostic par mois
print(f'{"Mois":>5s}  {"c":>6s}  {"scale":>8s}  {"shift":>8s}  {"KS_p":>6s}  {"n":>3s}')
for m in range(1, 13):
    p = spei_result['monthly_params'].get(m)
    if p is None:
        print(f'  {m:02d}    -- pas de fit --')
    else:
        c, scale, shift, ks_p, n_b = p
        print(f'  {m:02d}    {c:6.3f}  {scale:8.1f}  {shift:8.1f}  {ks_p:6.3f}  {n_b:3d}')

print()
# Comparaison juillet 2022
idx_jul22 = None
for i, t in enumerate(dates):
    if t.year == 2022 and t.month == 7:
        idx_jul22 = i; break
old_spei = spi_series[idx_jul22] if idx_jul22 is not None else np.nan
new_spei = spei_series[idx_jul22] if idx_jul22 is not None else np.nan
print(f'Paris-Saclay juillet 2022 :')
print(f'  SPI-12 (ancien)          : {old_spei:+.3f}')
print(f'  SPEI-12 (fit mensuel)    : {new_spei:+.3f}')

### 10.1b  Fix pixel côtier ERA5

ERA5 à 0.25° mélange terre et mer pour les pixels côtiers → PET diluée.
**Détection** : si PET du pixel central < 60 % du max voisin 3×3, on reroute
vers le gridpoint continental le plus proche.

Ref : Hersbach et al. (2020), QJRMS — land-sea mask fraction in ERA5.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 10.1b  Detection + correction pixel cotier
# ══════════════════════════════════════════════════════════════════════

def find_best_land_gridpoint(lat, lon, ds, month_idx=None,
                             threshold=0.60):
    """
    Detecte si le gridpoint ERA5 le plus proche est un pixel cotier
    (PET diluee par melange terre-mer) et reroute vers le voisin
    continental le plus representatif.
    
    Critere : PET_center < threshold * max(PET_3x3)
    
    Parameters
    ----------
    lat, lon : float
        Coordonnees du site.
    ds : xr.Dataset
        ERA5 ouvert (doit contenir 'pev').
    month_idx : int or None
        Indice temporel pour le test (defaut: juillet de la derniere annee).
    threshold : float
        Ratio min PET_center / PET_max_voisin (defaut 0.60).
    
    Returns
    -------
    best_lat, best_lon : float
        Gridpoint corrige (= original si non-cotier).
    is_coastal : bool
        True si pixel cotier detecte.
    ratio : float
        PET_center / PET_max_voisin.
    """
    td = 'valid_time' if 'valid_time' in ds.dims else 'time'
    lats = ds.latitude.values
    lons = ds.longitude.values
    
    # Trouver indices du gridpoint le plus proche
    i_lat = int(np.argmin(np.abs(lats - lat)))
    i_lon = int(np.argmin(np.abs(lons - lon)))
    
    # Indice temporel : juillet derniere annee par defaut
    if month_idx is None:
        times = pd.DatetimeIndex(ds[td].values)
        for k in range(len(times)-1, -1, -1):
            if times[k].month == 7:
                month_idx = k
                break
    
    # Extraire PET sur voisinage 3x3
    lat_slice = slice(max(0, i_lat-1), min(len(lats), i_lat+2))
    lon_slice = slice(max(0, i_lon-1), min(len(lons), i_lon+2))
    
    pev_3x3 = ds['pev'].isel({td: month_idx, 'latitude': lat_slice, 'longitude': lon_slice})
    pet_3x3 = np.maximum(-pev_3x3.values.astype(float), 0.0)  # PET = -pev (instantane)
    
    pet_center = pet_3x3[min(1, i_lat), min(1, i_lon)]
    pet_max = np.max(pet_3x3)
    
    ratio = pet_center / pet_max if pet_max > 0 else 1.0
    is_coastal = ratio < threshold
    
    if not is_coastal:
        return lats[i_lat], lons[i_lon], False, ratio
    
    # Trouver le voisin avec PET max (= plus continental)
    local_lats = lats[lat_slice]
    local_lons = lons[lon_slice]
    max_pos = np.unravel_index(np.argmax(pet_3x3), pet_3x3.shape)
    best_lat = float(local_lats[max_pos[0]])
    best_lon = float(local_lons[max_pos[1]])
    
    return best_lat, best_lon, True, ratio


# ── Diagnostic sur les 8 sites ──────────────────────────────────────
ds_check = xr.open_dataset(ERA5_FULL)

print('=== Detection pixels cotiers ERA5 ===\n')
print(f'{"Site":>14s}  {"lat_in":>7s}  {"lon_in":>7s}  '
      f'{"lat_out":>7s}  {"lon_out":>7s}  {"ratio":>6s}  Status')
print('-' * 75)

coastal_map = {}
for name, (lat, lon) in SITES.items():
    blat, blon, coastal, ratio = find_best_land_gridpoint(lat, lon, ds_check)
    coastal_map[name] = (blat, blon, coastal)
    status = 'COASTAL → fix' if coastal else 'OK'
    print(f'{name:>14s}  {lat:7.2f}  {lon:7.2f}  '
          f'{blat:7.2f}  {blon:7.2f}  {ratio:6.1%}  {status}')

ds_check.close()
print('\n→ coastal_map pret pour injection dans le modele v2.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 10.2  Modele v2 — multi-sites AVEC fix cotier
# ══════════════════════════════════════════════════════════════════════

WEIGHTS_V2 = {'w_b': 0.30, 'w_g': 0.20, 'w_d': 0.40, 'w_U': 0.10}

ds = xr.open_dataset(ERA5_FULL)
td = 'valid_time' if 'valid_time' in ds.dims else 'time'
times_era5 = pd.DatetimeIndex(ds[td].values)

rows_v2 = []
for name, (lat, lon) in SITES.items():
    bws = extract_raster_point(AQUEDUCT_BWS, lat, lon)
    gws = extract_raster_point(AQUEDUCT_GWS, lat, lon)
    
    # ── Fix cotier : rerouter PET si pixel mixte terre-mer ──
    pet_lat, pet_lon, is_coastal = coastal_map[name]
    
    # Precip : toujours au gridpoint original (moins sensible)
    tp = ds['tp'].sel(latitude=lat, longitude=lon, method='nearest')
    t = pd.DatetimeIndex(tp[td].values)
    dy = t.days_in_month.values.astype(float)
    p = np.maximum(tp.values.astype(float) * dy * 1000, 0)
    
    # PET : au gridpoint corrige si cotier
    pv = ds['pev'].sel(latitude=pet_lat, longitude=pet_lon, method='nearest')
    pet = np.maximum(-pv.values.astype(float) * dy * 1000, 0)
    
    sr = compute_spei_1d(p, pet, t)
    
    # July 2022
    idx = None
    for i, tt in enumerate(t):
        if tt.year == 2022 and tt.month == 7:
            idx = i; break
    
    b = float(np.clip(bws/5, 0, 1)) if not np.isnan(bws) else np.nan
    g = float(np.clip(gws/5, 0, 1)) if not np.isnan(gws) else np.nan
    spei_val = sr['spei'][idx]
    d = float(np.clip(-spei_val/3, 0, 1)) if not np.isnan(spei_val) else np.nan
    u_s = (0.5 if np.isnan(bws) else 0) + (0.5 if np.isnan(gws) else 0)
    u_f = float(np.clip(1-10*sr['ks_pvalue'], 0, 1))
    U = 0.5*u_s + 0.5*u_f
    
    h = np.array([b, g, d, U])
    w = np.array([0.30, 0.20, 0.40, 0.10])
    nm = np.isnan(h)
    if np.any(nm):
        wv = w.copy(); wv[nm] = 0; wv /= wv.sum()
        L = float(np.clip(np.dot(wv, np.where(nm, 0, h)), 0, 1))
    else:
        L = float(np.clip(np.dot(w, h), 0, 1))
    
    tag = ' *' if is_coastal else ''
    rows_v2.append({'Site': name, 'b': b, 'g': g, 'd_SPEI': d, 'U': U,
                    'L_v2': L, 'SPEI': spei_val,
                    'P': p[idx], 'PET': pet[idx], 'D': sr['D_monthly'][idx],
                    'coastal_fix': is_coastal})

ds.close()
df_v2 = pd.DataFrame(rows_v2).set_index('Site')

print('=== Modele v2 (SPEI) + fix cotier — Multi-sites, juillet 2022 ===\n')
print(df_v2.drop(columns=['coastal_fix']).round(3).to_string())

# Marquer les sites corriges
fixed = df_v2[df_v2['coastal_fix']].index.tolist()
if fixed:
    print(f'\n* Sites avec correction cotiere : {", ".join(fixed)}')

# Gradient NW→SE attendu
print('\n=== Gradient geographique ===')
nw = ['Brest', 'Rennes']
se = ['Marseille', 'Montpellier', 'Lyon']
L_nw = df_v2.loc[nw, 'L_v2'].mean()
L_se = df_v2.loc[se, 'L_v2'].mean()
print(f'  NW moyen (Brest, Rennes)          : L = {L_nw:.3f}')
print(f'  SE moyen (Marseille, Montpellier, Lyon) : L = {L_se:.3f}')
print(f'  Ratio SE/NW : {L_se/L_nw:.2f}x')
print(f'  → Gradient {"coherent ✓" if L_se > L_nw else "INCOHERENT ✗"}')

---
## 10.3  Validation multi-années

3 scenarios : **2015** (année normale), **2018** (canicule été), **2022** (sécheresse record).
On vérifie que le modèle discrimine bien les régimes hydrologiques.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 10.3  Validation multi-annees (2015, 2018, 2022)
# ══════════════════════════════════════════════════════════════════════

TEST_YEARS = [
    (2015, 7, "2015 — normale"),
    (2018, 7, "2018 — canicule"),
    (2022, 7, "2022 — secheresse"),
]

ds = xr.open_dataset(ERA5_FULL)
td = 'valid_time' if 'valid_time' in ds.dims else 'time'
times_all = pd.DatetimeIndex(ds[td].values)

all_results = []

for year, month, label in TEST_YEARS:
    rows = []
    for name, (lat, lon) in SITES.items():
        bws = extract_raster_point(AQUEDUCT_BWS, lat, lon)
        gws = extract_raster_point(AQUEDUCT_GWS, lat, lon)
        
        pet_lat, pet_lon, is_coastal = coastal_map[name]
        
        tp = ds['tp'].sel(latitude=lat, longitude=lon, method='nearest')
        t = pd.DatetimeIndex(tp[td].values)
        dy = t.days_in_month.values.astype(float)
        p = np.maximum(tp.values.astype(float) * dy * 1000, 0)
        
        pv = ds['pev'].sel(latitude=pet_lat, longitude=pet_lon, method='nearest')
        pet = np.maximum(-pv.values.astype(float) * dy * 1000, 0)
        
        sr = compute_spei_1d(p, pet, t)
        
        idx = None
        for i, tt in enumerate(t):
            if tt.year == year and tt.month == month:
                idx = i; break
        if idx is None:
            continue
        
        b = float(np.clip(bws/5, 0, 1)) if not np.isnan(bws) else np.nan
        g = float(np.clip(gws/5, 0, 1)) if not np.isnan(gws) else np.nan
        spei_val = sr['spei'][idx]
        d = float(np.clip(-spei_val/3, 0, 1)) if not np.isnan(spei_val) else np.nan
        u_s = (0.5 if np.isnan(bws) else 0) + (0.5 if np.isnan(gws) else 0)
        u_f = float(np.clip(1-10*sr['ks_pvalue'], 0, 1))
        U = 0.5*u_s + 0.5*u_f
        
        h = np.array([b, g, d, U])
        w = np.array([0.30, 0.20, 0.40, 0.10])
        nm = np.isnan(h)
        if np.any(nm):
            wv = w.copy(); wv[nm] = 0; wv /= wv.sum()
            L = float(np.clip(np.dot(wv, np.where(nm, 0, h)), 0, 1))
        else:
            L = float(np.clip(np.dot(w, h), 0, 1))
        
        rows.append({'Site': name, 'year': year, 'L': L,
                     'SPEI': spei_val, 'd': d, 'P': p[idx], 'PET': pet[idx]})
        all_results.append(rows[-1])
    
    df_yr = pd.DataFrame(rows).set_index('Site')
    print(f'=== {label} (juillet) ===')
    print(df_yr[['L', 'SPEI', 'd', 'P', 'PET']].round(3).to_string())
    print()

ds.close()

# ── Synthese comparative ────────────────────────────────────────────
df_all = pd.DataFrame(all_results)
pivot = df_all.pivot(index='Site', columns='year', values='L')

print('=== L(x,t) par site et par annee ===\n')
print(pivot.round(3).to_string())
print()

# Moyennes
for year, _, label in TEST_YEARS:
    mean_L = pivot[year].mean()
    print(f'  {label:25s} : L moyen = {mean_L:.3f}')

print()
# Test: 2022 > 2018 > 2015 ?
m15, m18, m22 = pivot[2015].mean(), pivot[2018].mean(), pivot[2022].mean()
ok = m22 > m18 > m15
print(f'Ordering L_2022 > L_2018 > L_2015 : {"✓" if ok else "✗"} ({m22:.3f} > {m18:.3f} > {m15:.3f})')

# Gradient NW-SE par annee
nw_sites = ['Brest', 'Rennes']
se_sites = ['Marseille', 'Montpellier', 'Lyon']
print()
print('=== Gradient NW→SE par annee ===')
for year, _, label in TEST_YEARS:
    sub = df_all[df_all['year'] == year].set_index('Site')
    nw = sub.loc[[s for s in nw_sites if s in sub.index], 'L'].mean()
    se = sub.loc[[s for s in se_sites if s in sub.index], 'L'].mean()
    grad = '✓' if se > nw else '✗'
    print(f'  {label:25s} : NW={nw:.3f}, SE={se:.3f}, ratio={se/nw:.2f}x  {grad}')

---
## 11. Modèle v3 — Structurel + Conjoncturel

Restructuration complète. Le score n'est plus un indice climatologique,
c'est un **score de viabilité hydrologique pour data center**.

$$H(x,t) = \underbrace{S_{\text{struct}}(x)}_{\text{climat + ressource}} + \underbrace{S_{\text{conj}}(x,t)}_{\text{anomalie crise}}$$

| Composante | Variable | Poids | Nature |
|---|---|---|---|
| Structurel | BWS (Aqueduct) | 0.35 | Pression sur la ressource |
| Structurel | Aridité (1 − P/PET) | 0.25 | Disponibilité absolue |
| Structurel | GWS (Aqueduct) | 0.10 | Stress souterrain |
| Conjoncturel | SPEI-12 → stress | 0.30 | Crise en cours |

**Output triple** : $S_{struct}$, $S_{conj}$, $H$ combiné.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 11.1  Aridity Index (transformation continue) + Modele v3
# ══════════════════════════════════════════════════════════════════════
#
# Correction : max(0, 1-AI) ecrase tout AI > 1 a zero.
# On utilise a la place une sigmoide calibree sur la classification UNEP :
#   AI < 0.20  hyper-aride   → stress ~ 1.0
#   AI = 0.50  semi-aride    → stress ~ 0.65
#   AI = 0.65  sub-humide sec→ stress ~ 0.50
#   AI = 1.00  humide        → stress ~ 0.20
#   AI = 1.50  tres humide   → stress ~ 0.05
#   AI > 2.00               → stress → 0
#
# Forme : stress = 1 / (1 + (AI / AI_mid)^k)
# avec AI_mid = 0.65 (seuil sub-humide), k = 3 (pente)

def aridity_stress_sigmoid(ai, ai_mid=0.65, k=3.0):
    """
    Stress d'aridite via sigmoide generalisee.
    Ref calibration : UNEP (1992), Middleton & Thomas (1997).
    """
    if ai <= 0:
        return 1.0
    return float(1.0 / (1.0 + (ai / ai_mid) ** k))

def compute_aridity_stress(precip_mm, pet_mm, dates,
                           baseline_start=1981, baseline_end=2010):
    """
    Calcule AI = P_annuel / PET_annuel sur baseline,
    puis transforme en stress via sigmoide.
    """
    years = dates.year
    mask = (years >= baseline_start) & (years <= baseline_end)
    p_base = precip_mm[mask]
    pet_base = pet_mm[mask]
    n_years = baseline_end - baseline_start + 1
    p_annual = np.sum(p_base) / n_years
    pet_annual = np.sum(pet_base) / n_years
    if pet_annual <= 0:
        return 0.0, p_annual, pet_annual, 999.0
    ai = p_annual / pet_annual
    stress = aridity_stress_sigmoid(ai)
    return stress, p_annual, pet_annual, ai

# ── Visualisation de la courbe sigmoide ─────────────────────────────
ai_range = np.linspace(0, 2.5, 200)
stress_sigmoid = [aridity_stress_sigmoid(a) for a in ai_range]
stress_linear = [float(np.clip(1.0 - a, 0, 1)) for a in ai_range]

print('=== Aridity stress : sigmoide vs lineaire ===')
print()
print(f'{"AI":>5s}  {"sigmoid":>8s}  {"linear":>8s}  Zone UNEP')
for ai_val, zone in [(0.05, 'hyper-aride'), (0.20, 'aride'),
                     (0.50, 'semi-aride'), (0.65, 'sub-humide sec'),
                     (1.00, 'humide'), (1.50, 'tres humide'),
                     (2.00, 'surplus')]:
    s_sig = aridity_stress_sigmoid(ai_val)
    s_lin = float(np.clip(1 - ai_val, 0, 1))
    print(f'{ai_val:5.2f}  {s_sig:8.3f}  {s_lin:8.3f}  {zone}')

# -- Poids v3 --
W_V3 = {'bws': 0.35, 'aridity': 0.25, 'gws': 0.10, 'spei': 0.30}
assert abs(sum(W_V3.values()) - 1.0) < 1e-9

TEST_DATES = [
    (2015, 7, "2015 -- normale"),
    (2018, 7, "2018 -- canicule"),
    (2022, 7, "2022 -- secheresse"),
]

ds = xr.open_dataset(ERA5_FULL)
td = 'valid_time' if 'valid_time' in ds.dims else 'time'

all_v3 = []
for name, (lat, lon) in SITES.items():
    bws = extract_raster_point(AQUEDUCT_BWS, lat, lon)
    gws = extract_raster_point(AQUEDUCT_GWS, lat, lon)
    pet_lat, pet_lon, is_coastal = coastal_map[name]
    tp = ds['tp'].sel(latitude=lat, longitude=lon, method='nearest')
    t = pd.DatetimeIndex(tp[td].values)
    dy = t.days_in_month.values.astype(float)
    p = np.maximum(tp.values.astype(float) * dy * 1000, 0)
    pv = ds['pev'].sel(latitude=pet_lat, longitude=pet_lon, method='nearest')
    pet = np.maximum(-pv.values.astype(float) * dy * 1000, 0)
    arid_stress, p_ann, pet_ann, ai = compute_aridity_stress(p, pet, t)
    sr = compute_spei_1d(p, pet, t)
    b = float(np.clip(bws / 5, 0, 1)) if not np.isnan(bws) else np.nan
    g = float(np.clip(gws / 5, 0, 1)) if not np.isnan(gws) else np.nan

    for year, month, label in TEST_DATES:
        idx = None
        for i, tt in enumerate(t):
            if tt.year == year and tt.month == month:
                idx = i; break
        if idx is None: continue
        spei_val = sr['spei'][idx]
        d_spei = float(np.clip(-spei_val / 3, 0, 1)) if not np.isnan(spei_val) else np.nan

        comps = {'bws': b, 'aridity': arid_stress, 'gws': g}
        s_struct = sum(W_V3[k] * v for k, v in comps.items()
                       if v is not None and not np.isnan(v))
        w_struct = sum(W_V3[k] for k, v in comps.items()
                       if v is not None and not np.isnan(v))
        s_conj = W_V3['spei'] * d_spei if (d_spei is not None and not np.isnan(d_spei)) else 0.0
        w_conj = W_V3['spei'] if (d_spei is not None and not np.isnan(d_spei)) else 0.0
        w_total = w_struct + w_conj
        if w_total > 0:
            H = float(np.clip((s_struct + s_conj) / w_total, 0, 1))
            s_struct_n = s_struct / w_struct if w_struct > 0 else np.nan
            s_conj_n = s_conj / w_conj if w_conj > 0 else np.nan
        else:
            H, s_struct_n, s_conj_n = np.nan, np.nan, np.nan

        all_v3.append({
            'Site': name, 'year': year, 'label': label,
            'S_struct': s_struct_n, 'S_conj': s_conj_n, 'H': H,
            'BWS': b, 'Aridity': arid_stress, 'GWS': g,
            'SPEI': spei_val, 'd_SPEI': d_spei,
            'AI': ai, 'P_ann': p_ann, 'PET_ann': pet_ann,
            'coastal': is_coastal})

ds.close()
df_v3 = pd.DataFrame(all_v3)

# -- Affichage par annee --
for year, _, label in TEST_DATES:
    sub = df_v3[df_v3['year'] == year].set_index('Site')
    print(f'\n=== {label} ===')
    print(sub[['S_struct', 'S_conj', 'H', 'BWS', 'Aridity', 'AI', 'SPEI']].round(3).to_string())

# -- Pivot H --
pivot = df_v3.pivot(index='Site', columns='year', values='H')
print('\n=== H(x,t) par site et par annee ===\n')
print(pivot.round(3).to_string())
print()
for year, _, label in TEST_DATES:
    print(f'  {label:25s} : H moyen = {pivot[year].mean():.3f}')
m15, m18, m22 = pivot[2015].mean(), pivot[2018].mean(), pivot[2022].mean()
print(f'\n  Ordering H_2022 > H_2018 > H_2015 : '
      f'{"OK" if m22 > m18 > m15 else "non"} ({m22:.3f} > {m18:.3f} > {m15:.3f})')

# -- Gradient NW->SE --
nw = ['Brest', 'Rennes']
se = ['Marseille', 'Montpellier', 'Lyon']
print('\n=== Gradient NW -> SE ===')
for year, _, label in TEST_DATES:
    sub = df_v3[df_v3['year'] == year].set_index('Site')
    nw_h = sub.loc[[s for s in nw if s in sub.index], 'H'].mean()
    se_h = sub.loc[[s for s in se if s in sub.index], 'H'].mean()
    grad = 'OK' if se_h > nw_h else 'non'
    print(f'  {label:25s} : NW={nw_h:.3f}, SE={se_h:.3f}, '
          f'ratio={se_h/nw_h:.2f}x  {grad}')

# -- Decomposition 2022 --
print('\n=== Decomposition S_struct vs S_conj (2022) ===')
sub22 = df_v3[df_v3['year'] == 2022].set_index('Site')
print(sub22[['S_struct', 'S_conj', 'H', 'Aridity', 'AI']].round(3).to_string())
print(f'\nS_struct moyen : {sub22["S_struct"].mean():.3f}')
print(f'S_conj moyen   : {sub22["S_conj"].mean():.3f}')

---
## 12. Diagnostics de robustesse

### 12.1 Qualité du fit SPEI — KS statistic par site × mois

On vérifie que la log-logistique (fisk) s'ajuste correctement
aux données de chaque mois calendaire, pour chaque site.
Seuil : KS p-value > 0.05 (on ne rejette pas H0).

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 12.1  KS diagnostic — fit SPEI par site x mois
# ══════════════════════════════════════════════════════════════════════

ds = xr.open_dataset(ERA5_FULL)
td = 'valid_time' if 'valid_time' in ds.dims else 'time'

ks_rows = []
for name, (lat, lon) in SITES.items():
    pet_lat, pet_lon, is_coastal = coastal_map[name]
    tp = ds['tp'].sel(latitude=lat, longitude=lon, method='nearest')
    t = pd.DatetimeIndex(tp[td].values)
    dy = t.days_in_month.values.astype(float)
    p = np.maximum(tp.values.astype(float) * dy * 1000, 0)
    pv = ds['pev'].sel(latitude=pet_lat, longitude=pet_lon, method='nearest')
    pet = np.maximum(-pv.values.astype(float) * dy * 1000, 0)
    sr = compute_spei_1d(p, pet, t)
    
    row = {'Site': name}
    n_fail = 0
    for m in range(1, 13):
        params = sr['monthly_params'].get(m)
        if params is None:
            row[f'm{m:02d}'] = np.nan
            n_fail += 1
        else:
            _, _, _, ks_p, _ = params
            row[f'm{m:02d}'] = ks_p
            if ks_p < 0.05:
                n_fail += 1
    row['n_reject'] = n_fail
    ks_rows.append(row)

ds.close()

df_ks = pd.DataFrame(ks_rows).set_index('Site')
month_cols = [f'm{m:02d}' for m in range(1, 13)]

print('=== KS p-values par site x mois (SPEI-12, fisk) ===')
print('    (< 0.05 = fit rejete)\n')
print(df_ks[month_cols + ['n_reject']].round(3).to_string())

total_tests = len(SITES) * 12
n_reject = int(df_ks['n_reject'].sum())
print(f'\nTotal : {n_reject}/{total_tests} fits rejetes a 5%')
if n_reject == 0:
    print('Tous les fits sont acceptes.')
elif n_reject <= total_tests * 0.05:
    print('Taux de rejet < 5% — acceptable (erreur de type I attendue).')
else:
    print(f'ATTENTION : {n_reject/total_tests:.0%} de rejets — verifier les sites concernes.')

### 12.2 Corrélation spatiale entre variables

Vérification du double comptage. Si deux variables sont fortement
corrélées (|r| > 0.7), l'une est redondante.
On calcule sur les 8 sites × 3 années = 24 observations.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 12.2  Matrice de correlation + VIF
# ══════════════════════════════════════════════════════════════════════

# Utiliser df_v3 deja calcule (8 sites x 3 annees)
vars_to_check = ['BWS', 'Aridity', 'GWS', 'd_SPEI']
df_corr_input = df_v3[vars_to_check].dropna()

print(f'=== Matrice de correlation (n={len(df_corr_input)}) ===\n')
corr = df_corr_input.corr()
print(corr.round(3).to_string())

# Paires problematiques
print('\n=== Paires avec |r| > 0.5 ===')
found = False
for i, v1 in enumerate(vars_to_check):
    for j, v2 in enumerate(vars_to_check):
        if j <= i: continue
        r = corr.loc[v1, v2]
        if abs(r) > 0.5:
            print(f'  {v1} x {v2} : r = {r:+.3f}  {"ATTENTION" if abs(r) > 0.7 else "a surveiller"}')
            found = True
if not found:
    print('  Aucune — pas de double comptage detecte.')

# VIF (Variance Inflation Factor)
print('\n=== VIF (>5 = multicolinearite) ===')
from numpy.linalg import inv
X = df_corr_input.values
X_centered = X - X.mean(axis=0)
xtx = X_centered.T @ X_centered
try:
    xtx_inv = inv(xtx)
    # VIF_j = diag(corr_inv)
    corr_mat = np.corrcoef(X_centered, rowvar=False)
    corr_inv = inv(corr_mat)
    for k, v in enumerate(vars_to_check):
        vif = corr_inv[k, k]
        flag = ' ATTENTION' if vif > 5 else ''
        print(f'  {v:10s} : VIF = {vif:.2f}{flag}')
except Exception as e:
    print(f'  VIF non calculable : {e}')

### 12.3 Harmonisation des échelles

Aujourd'hui :
- BWS, GWS : linéaire /5 (score Aqueduct)
- Aridity : sigmoïde sur ratio physique
- SPEI : clip linéaire /3

→ Les distributions effectives sur [0,1] sont très différentes.
Diagnostic : comparer les distributions et passer en **percentile rank**
pour que chaque variable contribue équitablement.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 12.3  Diagnostic des echelles + passage en percentile
# ══════════════════════════════════════════════════════════════════════

vars_diag = ['BWS', 'Aridity', 'GWS', 'd_SPEI']

print('=== Distribution effective des 4 variables (8 sites x 3 annees) ===\n')
print(f'{"Variable":>10s}  {"min":>6s}  {"p25":>6s}  {"med":>6s}  {"p75":>6s}  {"max":>6s}  {"std":>6s}  {"range_eff":>9s}')
print('-' * 72)

scale_issue = False
for v in vars_diag:
    vals = df_v3[v].dropna()
    p25, med, p75 = np.percentile(vals, [25, 50, 75])
    rng = vals.max() - vals.min()
    print(f'{v:>10s}  {vals.min():6.3f}  {p25:6.3f}  {med:6.3f}  {p75:6.3f}  {vals.max():6.3f}  {vals.std():6.3f}  {rng:9.3f}')
    if rng < 0.15:
        scale_issue = True

print()
if scale_issue:
    print('PROBLEME : certaines variables ont un range effectif tres faible.')
    print('Elles contribuent peu au score malgre leur poids nominal.')
else:
    print('Les ranges sont raisonnablement etales.')

# Transformation en percentile rank
print('\n=== Transformation percentile rank ===\n')
df_pctile = df_v3.copy()
for v in vars_diag:
    vals = df_pctile[v].values
    valid = ~np.isnan(vals)
    ranks = np.zeros_like(vals)
    ranks[valid] = stats.rankdata(vals[valid]) / np.sum(valid)
    ranks[~valid] = np.nan
    df_pctile[v + '_pct'] = ranks

# Comparer H avec poids percentile vs brut
pct_vars = [v + '_pct' for v in vars_diag]
weights = np.array([0.35, 0.25, 0.10, 0.30])

h_pct = []
for _, row in df_pctile.iterrows():
    vals = np.array([row[pv] for pv in pct_vars])
    nm = np.isnan(vals)
    if nm.all():
        h_pct.append(np.nan)
    else:
        w = weights.copy()
        w[nm] = 0
        w /= w.sum()
        h_pct.append(float(np.dot(w, np.where(nm, 0, vals))))
df_pctile['H_pct'] = h_pct

# Afficher la comparaison
comp = df_pctile[['Site', 'year', 'H', 'H_pct']].copy()
comp_pivot_h = comp.pivot(index='Site', columns='year', values='H')
comp_pivot_p = comp.pivot(index='Site', columns='year', values='H_pct')

print('H brut vs H percentile — 2022 :')
sub = comp[comp['year'] == 2022].set_index('Site')
print(sub[['H', 'H_pct']].round(3).to_string())

print('\nCorrelation H_brut vs H_pct :',
      f'{comp["H"].corr(comp["H_pct"]):.3f}')
print()
if comp['H'].corr(comp['H_pct']) > 0.90:
    print('Forte correlation — le classement est robuste au choix d echelle.')
    print('Recommendation : garder les scores physiques (plus interpretables),')
    print('mais documenter que le ranking est stable en percentile.')
else:
    print('ATTENTION : le classement change significativement.')
    print('Envisager de passer en percentile pour eviter les biais d echelle.')

### 12.4 SPEI-3 — volatilité conjoncturelle

SPEI-3 capte les chocs courts (canicule 2018).
On ne remplace pas SPEI-12, on compare les deux pour mesurer
la **volatilité** : un site où SPEI-3 << SPEI-12 subit un choc aigu.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 12.4  SPEI-3 — chocs courts + volatilite
# ══════════════════════════════════════════════════════════════════════

ds = xr.open_dataset(ERA5_FULL)
td = 'valid_time' if 'valid_time' in ds.dims else 'time'

spei_comp = []
for name, (lat, lon) in SITES.items():
    pet_lat, pet_lon, _ = coastal_map[name]
    tp = ds['tp'].sel(latitude=lat, longitude=lon, method='nearest')
    t = pd.DatetimeIndex(tp[td].values)
    dy = t.days_in_month.values.astype(float)
    p = np.maximum(tp.values.astype(float) * dy * 1000, 0)
    pv = ds['pev'].sel(latitude=pet_lat, longitude=pet_lon, method='nearest')
    pet = np.maximum(-pv.values.astype(float) * dy * 1000, 0)
    
    sr12 = compute_spei_1d(p, pet, t, window=12)
    sr3 = compute_spei_1d(p, pet, t, window=3)
    
    for year, month, label in [(2015, 7, '2015'), (2018, 7, '2018'), (2022, 7, '2022')]:
        idx = None
        for i, tt in enumerate(t):
            if tt.year == year and tt.month == month:
                idx = i; break
        if idx is None: continue
        
        s12 = sr12['spei'][idx]
        s3 = sr3['spei'][idx]
        delta = s3 - s12 if (not np.isnan(s3) and not np.isnan(s12)) else np.nan
        
        spei_comp.append({
            'Site': name, 'year': year,
            'SPEI_12': s12, 'SPEI_3': s3,
            'delta': delta,
        })

ds.close()
df_spei = pd.DataFrame(spei_comp)

# Affichage par annee
for year in [2015, 2018, 2022]:
    sub = df_spei[df_spei['year'] == year].set_index('Site')
    print(f'=== Juillet {year} : SPEI-12 vs SPEI-3 ===')
    print(sub[['SPEI_12', 'SPEI_3', 'delta']].round(3).to_string())
    
    # Interpretation
    acute = sub[sub['delta'] < -0.5]
    if len(acute) > 0:
        print(f'  → Choc aigu (SPEI-3 << SPEI-12) : {", ".join(acute.index)}')
    else:
        print(f'  → Pas de choc aigu detecte')
    print()

# Est-ce que SPEI-3 capture 2018 ?
print('=== Test : SPEI-3 capture-t-il la canicule 2018 ? ===')
sub18 = df_spei[df_spei['year'] == 2018].set_index('Site')
n_neg_12 = (sub18['SPEI_12'] < -0.5).sum()
n_neg_3 = (sub18['SPEI_3'] < -0.5).sum()
print(f'  Sites avec SPEI-12 < -0.5 : {n_neg_12}/8')
print(f'  Sites avec SPEI-3  < -0.5 : {n_neg_3}/8')
if n_neg_3 > n_neg_12:
    print(f'  → Oui, SPEI-3 detecte {n_neg_3 - n_neg_12} site(s) supplementaire(s)')
else:
    print(f'  → Non, SPEI-3 ne detecte pas plus de stress que SPEI-12 en 2018')

---
## 13. Modèle v4 — Correction structurelle

4 corrections simultanées :
1. **Scaling par variance** : $w_i^{eff} = w_i / \sigma_i$ (normalise la contribution réelle)
2. **Séparation mathématique** : $H = \alpha \cdot S_{struct} + (1-\alpha) \cdot S_{conj}$ avec poids intra-blocs
3. **SPEI** : $d = \max(f(\text{SPEI-12}), f(\text{SPEI-3}))$ (capte chocs courts)
4. **Décomposition** : contribution réelle (%) de chaque variable au score final

In [ ]:
# ======================================================================
# 13. Modele v4 -- corrections structurelles (v4b)
# ======================================================================
#
# Fix 1 : SPEI cappe a [-3, +3] avant transformation (au-dela = artefact fit)
# Fix 2 : BWS rescale sur range France [0, bws_france_max] au lieu de [0, 5]
#          Aqueduct score 5 = ratio WD/WS > 5x, jamais atteint en France
#          max observe = ~0.5 (BWS raw). On normalise / 0.5 au lieu de /5.
# Fix 3 : GWS idem, max France ~0.1, on normalise / 0.15 (avec marge)

# -- Ranges France (estimes sur les 8 sites + connaissance Aqueduct) --
BWS_MAX_FR = 0.60   # BWS raw max France ~ 0.5, marge 20%
GWS_MAX_FR = 0.15   # GWS raw max France ~ 0.1, marge 50%
SPEI_CAP = 3.0      # valeurs credibles [-3, +3]

print('=== Rescaling France ===')
print(f'  BWS normalise sur [0, {BWS_MAX_FR}] au lieu de [0, 5]')
print(f'  GWS normalise sur [0, {GWS_MAX_FR}] au lieu de [0, 5]')
print(f'  SPEI cappe a [{-SPEI_CAP}, +{SPEI_CAP}]')
print()

# -- Architecture v4 --
# Poids intra-struct (nominaux)
ALPHA = {'bws': 0.50, 'aridity': 0.36, 'gws': 0.14}
BETA = 0.70  # 70% structurel

def spei_to_stress(spei_val, cap=SPEI_CAP, scale=3.0):
    if np.isnan(spei_val): return np.nan
    capped = np.clip(spei_val, -cap, cap)
    return float(np.clip(-capped / scale, 0, 1))

ds = xr.open_dataset(ERA5_FULL)
td = 'valid_time' if 'valid_time' in ds.dims else 'time'

TEST_DATES = [
    (2015, 7, '2015 -- normale'),
    (2018, 7, '2018 -- canicule'),
    (2022, 7, '2022 -- secheresse'),
]

all_v4 = []
for name, (lat, lon) in SITES.items():
    bws_raw = extract_raster_point(AQUEDUCT_BWS, lat, lon)
    gws_raw = extract_raster_point(AQUEDUCT_GWS, lat, lon)
    pet_lat, pet_lon, is_coastal = coastal_map[name]
    tp = ds['tp'].sel(latitude=lat, longitude=lon, method='nearest')
    t = pd.DatetimeIndex(tp[td].values)
    dy = t.days_in_month.values.astype(float)
    p = np.maximum(tp.values.astype(float) * dy * 1000, 0)
    pv = ds['pev'].sel(latitude=pet_lat, longitude=pet_lon, method='nearest')
    pet = np.maximum(-pv.values.astype(float) * dy * 1000, 0)
    arid_stress, _, _, ai = compute_aridity_stress(p, pet, t)
    sr12 = compute_spei_1d(p, pet, t, window=12)
    sr3 = compute_spei_1d(p, pet, t, window=3)

    # Rescaling France
    b = float(np.clip(bws_raw / BWS_MAX_FR, 0, 1)) if not np.isnan(bws_raw) else np.nan
    g = float(np.clip(gws_raw / GWS_MAX_FR, 0, 1)) if not np.isnan(gws_raw) else np.nan

    for year, month, label in TEST_DATES:
        idx = None
        for i, tt in enumerate(t):
            if tt.year == year and tt.month == month:
                idx = i; break
        if idx is None: continue

        # S_struct : weighted mean (poids nominaux, pas variance-scaled)
        # Apres rescaling France, les ranges sont comparables
        comps = {'bws': b, 'aridity': arid_stress, 'gws': g}
        num, den = 0.0, 0.0
        for k, v in comps.items():
            if v is not None and not np.isnan(v):
                num += ALPHA[k] * v
                den += ALPHA[k]
        s_struct = num / den if den > 0 else np.nan

        # S_conj : max(stress_12, stress_3) avec cap
        d12 = spei_to_stress(sr12['spei'][idx])
        d3 = spei_to_stress(sr3['spei'][idx])
        if np.isnan(d12) and np.isnan(d3):
            s_conj = np.nan
        elif np.isnan(d3):
            s_conj = d12
        elif np.isnan(d12):
            s_conj = d3
        else:
            s_conj = max(d12, d3)

        # H = beta * S_struct + (1-beta) * S_conj
        if np.isnan(s_struct) and np.isnan(s_conj):
            H = np.nan
        elif np.isnan(s_struct):
            H = s_conj
        elif np.isnan(s_conj):
            H = s_struct
        else:
            H = BETA * s_struct + (1 - BETA) * s_conj
        H = float(np.clip(H, 0, 1)) if not np.isnan(H) else np.nan

        # Contributions reelles
        c_struct = BETA * s_struct if not np.isnan(s_struct) else 0.0
        c_conj = (1 - BETA) * s_conj if not np.isnan(s_conj) else 0.0
        c_total = c_struct + c_conj
        pct_s = c_struct / c_total * 100 if c_total > 0 else np.nan
        pct_c = c_conj / c_total * 100 if c_total > 0 else np.nan

        all_v4.append({
            'Site': name, 'year': year,
            'S_struct': s_struct, 'S_conj': s_conj, 'H': H,
            'BWS': b, 'Aridity': arid_stress, 'GWS': g,
            'SPEI12': sr12['spei'][idx], 'SPEI3': sr3['spei'][idx],
            'd12': d12, 'd3': d3,
            'AI': ai, '%_struct': pct_s, '%_conj': pct_c})

ds.close()
df_v4 = pd.DataFrame(all_v4)

# -- Distribution des variables rescalees --
print('=== Distribution apres rescaling France ===')
print(f'{"Variable":>10s}  {"min":>6s}  {"p25":>6s}  {"med":>6s}  {"p75":>6s}  {"max":>6s}  {"range":>6s}')
for v in ['BWS', 'Aridity', 'GWS', 'd12', 'd3', 'S_conj']:
    vals = df_v4[v].dropna()
    if len(vals) == 0: continue
    p25, med, p75 = np.percentile(vals, [25, 50, 75])
    print(f'{v:>10s}  {vals.min():6.3f}  {p25:6.3f}  {med:6.3f}  {p75:6.3f}  {vals.max():6.3f}  {vals.max()-vals.min():6.3f}')
print()

# -- Affichage par annee --
for year, _, label in TEST_DATES:
    sub = df_v4[df_v4['year'] == year].set_index('Site')
    print(f'=== {label} ===')
    print(sub[['S_struct', 'S_conj', 'H', '%_struct', '%_conj']].round(3).to_string())
    print()

# Pivot H
pivot = df_v4.pivot(index='Site', columns='year', values='H')
print('=== H(x,t) v4 par site et par annee ===\n')
print(pivot.round(3).to_string())
print()
for year, _, label in TEST_DATES:
    print(f'  {label:25s} : H moyen = {pivot[year].mean():.3f}')
m15, m18, m22 = pivot[2015].mean(), pivot[2018].mean(), pivot[2022].mean()
print(f'\n  Ordering : {"OK" if m22 > m18 > m15 else "non"} '
      f'({m22:.3f} > {m18:.3f} > {m15:.3f})')

# Gradient
nw = ['Brest', 'Rennes']
se = ['Marseille', 'Montpellier', 'Lyon']
print('\n=== Gradient NW -> SE ===')
for year, _, label in TEST_DATES:
    sub = df_v4[df_v4['year'] == year].set_index('Site')
    nw_h = sub.loc[[s for s in nw if s in sub.index], 'H'].mean()
    se_h = sub.loc[[s for s in se if s in sub.index], 'H'].mean()
    print(f'  {label:25s} : NW={nw_h:.3f}, SE={se_h:.3f}, '
          f'ratio={se_h/nw_h:.2f}x  {"OK" if se_h > nw_h else "non"}')

# Contribution
print('\n=== Contribution reelle moyenne (%) ===')
for year, _, label in TEST_DATES:
    sub = df_v4[df_v4['year'] == year]
    ps = sub['%_struct'].mean()
    pc = sub['%_conj'].mean()
    bal = 'EQUILIBRE' if 35 < ps < 65 else ('STRUCT domine' if ps > 65 else 'CONJ domine')
    print(f'  {label:25s} : struct={ps:.0f}%, conj={pc:.0f}%  {bal}')


---
## 0. Setup